In [2]:
# ─── 1. IMPORT LIBRARIES & SETUP ────────────────────────────────────────────
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# ─── 2. LOAD PRETRAINED BASE MODELS ────────────────────────────────────────
# === Load Models ===
model_x = load_model("xception_deepfake.h5")
model_v = load_model("vgg19.h5")
model_e = load_model("efficientnetb0.h5")
model_r = load_model("resnet50.h5")
model_m = load_model("mobilenetv2.h5")

data_dir = r"C:\Users\Admin\Downloads\Compressed\FaceForensics++\data_flat"


# ─── 3. PREPARE TEST DATA GENERATOR ───────────────────────────────────────
# === Generator for 224x224 models ===
datagen_224 = ImageDataGenerator(rescale=1./255, validation_split=0.2)

test_gen_224 = datagen_224.flow_from_directory(
    data_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary',
    subset='validation',
    shuffle=False
)

# === Generator for 299x299 (for Xception) ===
datagen_299 = ImageDataGenerator(rescale=1./255, validation_split=0.2)

test_gen_299 = datagen_299.flow_from_directory(
    data_dir,
    target_size=(299, 299),
    batch_size=32,
    class_mode='binary',
    subset='validation',
    shuffle=False
)

# ─── 4. GENERATE BASE MODEL PREDICTIONS ───────────────────────────────────
# === Predictions ===
pred_x = model_x.predict(test_gen_299, verbose=1)
pred_v = model_v.predict(test_gen_224, verbose=1)
pred_e = model_e.predict(test_gen_224, verbose=1)
pred_r = model_r.predict(test_gen_224, verbose=1)
pred_m = model_m.predict(test_gen_224, verbose=1)

# ─── 5. SOFT VOTING ENSEMBLE ──────────────────────────────────────────────
# Average probabilities across all five models
final_probs = (pred_x + pred_v + pred_e + pred_r + pred_m) / 5
final_preds = (final_probs > 0.5).astype(int)

# ─── 6. EVALUATE ENSEMBLE PERFORMANCE ────────────────────────────────────
true_labels = test_gen_224.classes  # both have same order since shuffle=False
print("\n✅ Ensemble Accuracy:", accuracy_score(true_labels, final_preds))
print("\n📊 Classification Report:\n", classification_report(true_labels, final_preds, target_names=["Fake", "Real"]))
print("\n🧾 Confusion Matrix:\n", confusion_matrix(true_labels, final_preds))


Found 2112 images belonging to 2 classes.
Found 2112 images belonging to 2 classes.
66/66 [==============================] - 34s 508ms/step

✅ Ensemble Accuracy: 0.47017045454545453

📊 Classification Report:
               precision    recall  f1-score   support

        Fake       0.45      0.70      0.55       966
        Real       0.52      0.28      0.36      1146

    accuracy                           0.47      2112
   macro avg       0.49      0.49      0.46      2112
weighted avg       0.49      0.47      0.45      2112


🧾 Confusion Matrix:
 [[672 294]
 [825 321]]
